# HMP-GNN — Google Colab

**仅主实验**（参数完全来自 `main.py` 中 `main()` 的 `config`）。

1. GPU：Runtime → Change runtime type → T4 等。
2. **Step 0** 取代码 → **Step 1** 安装依赖 → **Step 2** 检查环境。
3. **Step 3** 运行实验（可能耗时较长）。
4. **Step 4a** 实验摘要卡 + 全局指标三联图（acc / loss / CSE）+ PPL 摘要 — **全部 inline 渲染，断开运行时后仍在屏幕。**
5. **Step 4b** Per-client 过程图（7 张 inline 图）：local acc、local CSE、trust weights、graph residual、cosine similarity、euclidean distance、acc-vs-CSE 散点。
6. **Step 5** 数值明细（供复盘核查）：完整 config、逐轮全局指标表、逐轮 per-client trust/residual 表、逐轮 per-client acc/CSE 表。
7. **Step 6** 可选 zip 下载产物；**Step 7** 释放 GPU。

> **使用方式**：Run All → 等待实验完成 → 运行 Step 7 释放 GPU → 所有图表已嵌入 notebook 输出，可右键保存各图。


## Step 0: 获取代码

若当前目录下已有 `main.py`，不克隆；否则拉取并进入 `HMP-GNN`。

In [ ]:
# Fetch repository and set working directory
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/sileneer/HMP-GNN.git"
REPO_DIR = Path("HMP-GNN")

def code_files_present():
    return Path("main.py").is_file() and Path("data_loader.py").is_file()

if code_files_present():
    print("已在仓库根目录（发现 main.py）。")
else:
    if REPO_DIR.is_dir():
        print(f"使用已有目录: {REPO_DIR}")
    else:
        print(f"正在克隆 {REPO_URL} ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    os.chdir(REPO_DIR)
    print(f"工作目录: {Path('.').resolve()}")

sys.path.insert(0, str(Path('.').resolve()))
print("cwd:", Path('.').resolve())


## Step 1: 安装依赖

In [ ]:
from pathlib import Path
import subprocess
import sys

req = Path("requirements.txt")
if req.is_file():
    print("Installing from requirements.txt ...")
    %pip install -q -r requirements.txt
else:
    %pip install -q torch transformers datasets numpy scikit-learn pandas tqdm matplotlib seaborn peft "torchao>=0.17"
# Colab 常预装 torchao 0.10；PEFT 注入 LoRA 时若版本 <0.16 会直接 ImportError
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "torchao>=0.17"],
    check=False,
)
print("依赖已安装")


## Step 2: 检查文件与 GPU

In [ ]:
import os
from pathlib import Path
import torch

required = [
    "main.py", "client.py", "server.py", "data_loader.py", "models.py", "visualization.py",
    "defense/__init__.py", "fed_checkpoint.py", "evaluation_hallucination.py",
    "attack/hallucination.py", "attack/__init__.py", "hmp_gae/__init__.py",
]
missing = [f for f in required if not os.path.exists(f)]
if missing:
    print("缺少文件:", missing)
else:
    print("核心文件已就绪")
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Step 3: 运行主实验

**参数完全来自 `main.py` 中 `main()` 的 `config`**，本 notebook 不再做任何覆写。

In [ ]:
import os
import gc
import warnings
import torch
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

from main import main

print("=" * 60)
print("SINGLE 模式：直接使用 main.py 默认 config")
print("=" * 60)
try:
    main()
    print("\n实验完成")
except Exception as e:
    print("\n失败:", e)
    import traceback
    traceback.print_exc()
finally:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("gc + CUDA empty_cache 已执行")


## Step 4a: 实验摘要 + 全局指标

- **实验摘要卡**：关键配置参数（模型、数据集、攻击/防御方式、客户端数量）+ 最终结果（final acc、best acc、final CSE）。
- **全局指标三联图**（inline matplotlib，不读磁盘 PNG）：Global Accuracy / Global Loss / Global CSE over rounds。
- **PPL 摘要**：`eval_perplexity` 结果（需 decoder 骨干 + `save_global_checkpoint=True`）。

> 所有内容均为 inline 渲染，断开 GPU 运行时后仍保留在 notebook 输出，可右键保存图片。


In [ ]:
from IPython.display import display, Markdown
from pathlib import Path
import json
import matplotlib.pyplot as plt

# ========== IEEE publication-quality style ==========
plt.style.use("default")
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans", "Liberation Sans", "Helvetica", "sans-serif"],
    "font.size": 10,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "legend.frameon": True,
    "legend.framealpha": 1.0,
    "legend.fancybox": False,
    "legend.edgecolor": "black",
    "legend.borderpad": 0.4,
    "figure.titlesize": 12,
    "grid.linewidth": 0.5,
    "grid.alpha": 0.36,
    "lines.linewidth": 1.5,
    "lines.markersize": 5,
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
})

# IEEE-style per-metric colors + attack-start vertical marker
IEEE_BLUE  = "#0066CC"  # Global Accuracy
IEEE_RED   = "#C00000"  # Global Loss
IEEE_GREEN = "#00B050"  # Global CSE
IEEE_ATK_MK = "#C0392B"

R = Path("results")
FIG_DIR = R / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

candidates = sorted(R.glob("*_results.json"), key=lambda p: p.stat().st_mtime, reverse=True)
if not candidates:
    display(Markdown("**未找到 `results/*_results.json`，请先运行 Step 3。**"))
else:
    rpath = candidates[0]
    with open(rpath, "r", encoding="utf-8") as f:
        data = json.load(f)

    cfg         = data.get("config", {})
    exp         = cfg.get("experiment_name", "experiment")
    pm          = data.get("progressive_metrics", {})
    rounds_data = data.get("results", [])
    atk_ids     = data.get("attacker_ids", [])

    # ── helpers ──────────────────────────────────────────────────────────────
    def _last(lst):
        vals = [v for v in (lst or []) if isinstance(v, (int, float)) and not isinstance(v, bool)]
        return vals[-1] if vals else None

    def _mean(lst):
        vals = [v for v in (lst or []) if isinstance(v, (int, float)) and not isinstance(v, bool)]
        return sum(vals) / len(vals) if vals else None

    def _fmt(v, digits=4):
        return f"{v:.{digits}f}" if isinstance(v, (int, float)) else "—"

    clean_acc_list = pm.get("clean_acc", [])
    cse_list       = pm.get("cse", [])
    final_acc  = _last(clean_acc_list)
    best_acc   = max((v for v in clean_acc_list if isinstance(v, (int, float))), default=None)
    final_cse  = _last(cse_list)
    mean_cse   = _mean(cse_list)

    # ── ① 实验摘要卡 ─────────────────────────────────────────────────────────
    display(Markdown(f"## Step 4a 实验摘要 — `{exp}`"))

    lora_info = f"r={cfg.get('lora_r')} α={cfg.get('lora_alpha')}" if cfg.get("use_lora") else "False"
    atk_str   = str(atk_ids) if atk_ids else f"last {cfg.get('num_attackers',0)} clients (inferred)"
    flip_info = (f"{cfg.get('hallu_flip_mode','?')}  ratio={cfg.get('hallu_flip_ratio','?')}"
                 if cfg.get("attack_method") == "Hallucination" else "—")

    card_rows = [
        ("experiment_name",  exp),
        ("model_name",       cfg.get("model_name", "—")),
        ("dataset",          cfg.get("dataset", "—")),
        ("num_labels",       cfg.get("num_labels", "—")),
        ("use_lora",         lora_info),
        ("num_clients",      cfg.get("num_clients", "—")),
        ("num_attackers",    cfg.get("num_attackers", "—")),
        ("attacker_ids",     atk_str),
        ("attack_method",    cfg.get("attack_method", "—")),
        ("flip_mode / ratio",flip_info),
        ("defense_method",   cfg.get("defense_method", "—")),
        ("num_rounds",       cfg.get("num_rounds", "—")),
        ("data_distribution",cfg.get("data_distribution", "—")),
        ("dataset_size_limit",cfg.get("dataset_size_limit", "—")),
    ]
    result_rows = [
        ("**Final Accuracy**", f"**{_fmt(final_acc)}**"),
        ("**Best Accuracy**",  f"**{_fmt(best_acc)}**"),
        ("**Final CSE**",      f"**{_fmt(final_cse)}**"),
        ("**Mean CSE**",       f"**{_fmt(mean_cse)}**"),
    ]

    md = "| 参数 | 值 |\n|:---|:---|\n"
    for k, v in card_rows:
        md += f"| {k} | `{v}` |\n"
    md += "| — | — |\n"
    for k, v in result_rows:
        md += f"| {k} | {v} |\n"
    display(Markdown(md))

    # ── ② Global metrics — 3 individual IEEE-style figures (split for clarity) ──
    display(Markdown("### 全局指标（每张图独立 dpi=600 渲染并保存到 `results/figures/`）"))

    rounds = [r.get("round", i + 1) for i, r in enumerate(rounds_data)]
    acc    = [r.get("clean_accuracy", None)                  for r in rounds_data]
    loss   = [r.get("global_loss", None)                     for r in rounds_data]
    cse    = [r.get("classification_semantic_entropy", None) for r in rounds_data]

    atk_start = cfg.get("attack_start_round") or cfg.get("hallu_attack_start_round")

    def _plot_single(xs, ys, color, ylabel, title, fname, ylim=None, marker="o"):
        pairs = [(x, y) for x, y in zip(xs, ys) if y is not None]
        if not pairs:
            display(Markdown(f"*{title} — 无数据*"))
            return
        px, py = zip(*pairs)
        fig, ax = plt.subplots(figsize=(6.5, 5))
        me = max(1, len(px) // 25)
        ax.plot(px, py, marker=marker, linestyle="-", color=color,
                linewidth=1.8, markersize=5.5,
                markevery=me, zorder=3,
                markerfacecolor=color, markeredgecolor="white", markeredgewidth=0.6)
        if atk_start is not None and int(atk_start) > 0:
            ax.axvline(x=int(atk_start), color=IEEE_ATK_MK,
                       linewidth=1.0, linestyle=":", alpha=0.85,
                       label=f"attack start (r={int(atk_start)})", zorder=2)
            ax.legend(loc="best")
        ax.set_xlabel("Communication Round")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(True, alpha=0.36, linestyle="--", linewidth=0.5)
        for s in ("top", "right", "bottom", "left"):
            ax.spines[s].set_visible(True)
        if ylim is not None:
            ax.set_ylim(*ylim)
        ax.set_xlim(min(px), max(px))
        fig.tight_layout()
        out = FIG_DIR / f"{exp}__{fname}.png"
        fig.savefig(out)
        display(fig)
        plt.close(fig)
        display(Markdown(f"*Saved:* `{out}`"))

    _plot_single(rounds, acc,  IEEE_BLUE,  "Global Accuracy",
                 f"Global Accuracy — {exp}", "global_accuracy",
                 ylim=(0.0, 1.0), marker="o")
    _plot_single(rounds, loss, IEEE_RED,   "Global Loss",
                 f"Global Loss — {exp}",     "global_loss", marker="s")
    _plot_single(rounds, cse,  IEEE_GREEN, "CSE (nats)",
                 f"Global CSE — {exp}",      "global_cse",  marker="^")

    # ── ③ Perplexity (PPL) — global model on test subset ────────────────────
    # Shows everything eval_hallucination saves into <exp>_eval_ppl.json:
    #   ppl_mean / mean_nll / n_samples / model_name / dataset / max_length / seed
    #   ppl_per_class  (one PPL per stratified test class)
    # Per-class spread is a *defense uniformity* indicator: low spread (<5)
    # means HMP-GAE filters attackers symmetrically across classes; large
    # spread means some target classes still leak through.
    display(Markdown("### Perplexity (PPL) — Global Model"))
    ppl_path = R / f"{exp}_eval_ppl.json"
    if ppl_path.is_file():
        with open(ppl_path, "r", encoding="utf-8") as f:
            ppl = json.load(f)
        if ppl.get("skipped"):
            display(Markdown(f"*PPL 已跳过：* `{ppl.get('skip_reason', '?')}`"))
        else:
            mean_ppl = ppl.get("ppl_mean")
            mean_nll = ppl.get("mean_nll")
            per_cls  = ppl.get("ppl_per_class", {}) or {}

            display(Markdown(
                f"**PPL mean:** `{_fmt(mean_ppl, 2)}` &nbsp;|&nbsp; "
                f"**mean NLL:** `{_fmt(mean_nll, 4)}` &nbsp;|&nbsp; "
                f"**n_samples:** `{ppl.get('n_samples')}` &nbsp;|&nbsp; "
                f"**model:** `{ppl.get('model_name', cfg.get('model_name'))}` &nbsp;|&nbsp; "
                f"**dataset:** `{ppl.get('dataset', cfg.get('dataset'))}` &nbsp;|&nbsp; "
                f"**max_length:** `{ppl.get('max_length', cfg.get('max_length'))}` &nbsp;|&nbsp; "
                f"**seed:** `{ppl.get('seed', cfg.get('ppl_seed'))}`"
            ))

            if per_cls:
                # ag_news class labels (only used cosmetically; falls back to id)
                ag_names = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
                ds_name = (ppl.get("dataset") or cfg.get("dataset") or "").lower()

                # Sort by integer class id (keys come back as strings from JSON).
                items = sorted(per_cls.items(), key=lambda kv: int(kv[0]))
                vals  = [v for _, v in items if isinstance(v, (int, float))]
                pc_mean   = (sum(vals) / len(vals)) if vals else None
                pc_spread = (max(vals) - min(vals)) if len(vals) >= 2 else 0.0
                pc_std    = (
                    (sum((v - pc_mean) ** 2 for v in vals) / len(vals)) ** 0.5
                    if vals and pc_mean is not None else 0.0
                )

                md = "**Per-class PPL** (lower = less hallucination in this class)\n\n"
                md += "| Class | Label | PPL | Δ vs mean |\n|---:|:---|---:|---:|\n"
                for k, v in items:
                    label = ag_names.get(int(k), "—") if ds_name == "ag_news" else "—"
                    if isinstance(v, (int, float)) and pc_mean is not None:
                        delta = f"{v - pc_mean:+.2f}"
                        md += f"| {k} | {label} | {v:.2f} | {delta} |\n"
                    else:
                        md += f"| {k} | {label} | — | — |\n"
                md += (
                    f"| **mean** | — | **{_fmt(pc_mean, 2)}** | — |\n"
                    f"| **spread (max−min)** | — | **{pc_spread:.2f}** | — |\n"
                    f"| **std**              | — | **{pc_std:.2f}** | — |\n"
                )
                display(Markdown(md))
                display(Markdown(
                    "*Per-class spread is a defense-uniformity check: low spread "
                    "(<5 PPL) means HMP-GAE filters attackers symmetrically; "
                    "high spread suggests certain target classes still leak.*"
                ))
            else:
                display(Markdown("*`ppl_per_class` 字段为空。*"))
    else:
        display(Markdown(
            f"*未找到 `{ppl_path.name}`。*  "
            f"当前 config：`eval_perplexity={cfg.get('eval_perplexity')}`，"
            f"`save_global_checkpoint={cfg.get('save_global_checkpoint')}`。"
        ))


In [ ]:
from IPython.display import display, Markdown
from pathlib import Path
import json
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

# ========== IEEE publication-quality style ==========
plt.style.use("default")
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans", "Liberation Sans", "Helvetica", "sans-serif"],
    "font.size": 10,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "legend.frameon": True,
    "legend.framealpha": 1.0,
    "legend.fancybox": False,
    "legend.edgecolor": "black",
    "legend.borderpad": 0.4,
    "figure.titlesize": 12,
    "grid.linewidth": 0.5,
    "grid.alpha": 0.36,
    "lines.linewidth": 1.5,
    "lines.markersize": 5,
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
})

# IEEE color palette: distinct blues / oranges / greens for benign, reds for attackers
IEEE_BENIGN_COLORS = [
    "#0066CC", "#FF6600", "#00B050", "#FFC000", "#7030A0",
    "#C55A11", "#70AD47", "#5B9BD5", "#2E75B6", "#0070C0",
    "#954F72", "#1F4E79", "#000000", "#1C7C54", "#3B7EA1",
]
IEEE_ATTACKER_COLORS = [
    "#C00000", "#DC143C", "#FF4500", "#B22222", "#E74C3C",
    "#C0392B", "#8B0000", "#A52A2A",
]

display(Markdown("## Step 4b: Per-client 过程指标 (IEEE-style, 单图独立保存)"))
display(Markdown(
    "所有图表为 **inline matplotlib 渲染**（不读磁盘 PNG），"
    "断开 Colab 运行时后仍保留在 notebook 输出中。"
    "同时按 IEEE 出版品质 (`dpi=600`) 保存到 `results/figures/`。"
))

R = Path("results")
FIG_DIR = R / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

res_files = sorted(R.glob("*_results.json"), key=lambda p: p.stat().st_mtime, reverse=True)
if not res_files:
    display(Markdown("**未找到结果文件，请先运行 Step 3。**"))
else:
    rpath = res_files[0]
    with open(rpath, "r", encoding="utf-8") as f:
        data = json.load(f)

    cfg         = data.get("config", {})
    exp         = cfg.get("experiment_name", "experiment")
    rounds_data = data.get("results", [])

    # ── attacker IDs：直接从 JSON 读，不再推断 ──────────────────────────────
    attacker_ids = set(int(x) for x in (data.get("attacker_ids") or []))
    if not attacker_ids:
        nc, na = int(cfg.get("num_clients", 0)), int(cfg.get("num_attackers", 0))
        if na > 0 and nc > 0:
            attacker_ids = set(range(nc - na, nc))

    def is_atk(cid): return int(cid) in attacker_ids

    def _build_color_map(cids):
        m = {}
        b_idx = a_idx = 0
        for cid in sorted(cids):
            if is_atk(cid):
                m[cid] = IEEE_ATTACKER_COLORS[a_idx % len(IEEE_ATTACKER_COLORS)]
                a_idx += 1
            else:
                m[cid] = IEEE_BENIGN_COLORS[b_idx % len(IEEE_BENIGN_COLORS)]
                b_idx += 1
        return m

    def c_ls(cid):    return "--" if is_atk(cid) else "-"
    def c_label(cid): return f"C{cid}[ATK]" if is_atk(cid) else f"C{cid}[BGN]"

    # ── 收集逐轮逐 client 数据 ───────────────────────────────────────────────
    round_nums  = []
    local_acc   = {}
    local_cse   = {}
    trust_w     = {}
    graph_res   = {}
    cosine_sim  = {}
    euclid_dist = {}

    top_acc = data.get("local_accuracies") or {}
    top_cse = data.get("local_cse") or {}

    for r in rounds_data:
        rnum = r.get("round", len(round_nums) + 1)
        round_nums.append(rnum)
        for raw, v in (r.get("local_accuracies") or {}).items():
            cid = int(raw) if str(raw).isdigit() else raw
            local_acc.setdefault(cid, []).append(v)
        for raw, v in (r.get("local_cse") or {}).items():
            cid = int(raw) if str(raw).isdigit() else raw
            local_cse.setdefault(cid, []).append(v)
        agg      = r.get("aggregation") or {}
        c_order  = [int(x) if str(x).isdigit() else x for x in (agg.get("accepted_clients") or [])]
        alphas   = agg.get("trust_weights") or []
        residual = agg.get("residual") or []
        sims     = agg.get("similarities") or []
        dists    = agg.get("euclidean_distances") or []
        for i, cid in enumerate(c_order):
            if alphas   and i < len(alphas):   trust_w.setdefault(cid, []).append(alphas[i])
            if residual and i < len(residual): graph_res.setdefault(cid, []).append(residual[i])
            if sims     and i < len(sims):     cosine_sim.setdefault(cid, []).append(sims[i])
            if dists    and i < len(dists):    euclid_dist.setdefault(cid, []).append(dists[i])

    if top_acc:
        local_acc = {(int(k) if str(k).isdigit() else k): v for k, v in top_acc.items()}
    if top_cse:
        local_cse = {(int(k) if str(k).isdigit() else k): v for k, v in top_cse.items()}

    all_cids = sorted(set(list(local_acc) + list(local_cse) + list(trust_w)
                          + list(graph_res) + list(cosine_sim) + list(euclid_dist)))
    cmap = _build_color_map(all_cids)

    # ── 通用绘图函数（IEEE 单图，dpi=600 保存） ──────────────────────────────
    def _plot(series_dict, ylabel, title, marker, fname, note="", ylim=None):
        if not series_dict:
            display(Markdown(f"*{title} — 无数据（当前 defense 模式可能不产生此指标）*"))
            return
        fig, ax = plt.subplots(figsize=(7.0, 4.6))
        max_len = max((len(v) for v in series_dict.values()), default=1)
        me = max(1, max_len // 25)
        per_client_h = []
        for cid in sorted(series_dict):
            ys = series_dict[cid]
            xs = round_nums[:len(ys)]
            color = cmap.get(cid, "#666666")
            ls = c_ls(cid)
            ax.plot(xs, ys, marker=marker, markersize=4.5, lw=1.6,
                    color=color, ls=ls,
                    alpha=0.92, markevery=me,
                    markerfacecolor=color,
                    markeredgecolor="white", markeredgewidth=0.5)
            per_client_h.append(
                mlines.Line2D([], [], color=color, lw=1.6, ls=ls,
                              marker=marker, markersize=4.5,
                              markerfacecolor=color,
                              markeredgecolor="white", markeredgewidth=0.5,
                              label=c_label(cid))
            )
        ax.set_xlabel("Communication Round")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(True, alpha=0.36, linestyle="--", linewidth=0.5)
        for s in ("top", "right", "bottom", "left"):
            ax.spines[s].set_visible(True)
        ax.legend(handles=per_client_h,
                  ncol=2 if len(per_client_h) > 5 else 1,
                  loc="best", framealpha=1.0, edgecolor="black")
        if ylim:
            ax.set_ylim(*ylim)
        if round_nums:
            ax.set_xlim(min(round_nums), max(round_nums))
        fig.tight_layout()
        if note:
            display(Markdown(note))
        out = FIG_DIR / f"{exp}__{fname}.png"
        fig.savefig(out)
        display(fig)
        plt.close(fig)
        display(Markdown(f"*Saved:* `{out}`"))

    # ── ① Per-client Local Accuracy ──────────────────────────────────────────
    display(Markdown("---\n#### ① Per-client Local Accuracy"))
    _plot(local_acc, "Local Accuracy", "Per-client Local Accuracy",
          "o", "local_accuracy",
          note="local model 在 server test set 上的分类准确率。"
               "**Benign 应随训练上升；Attacker (pairwise flip) 在 clean test set 上准确率低。**",
          ylim=(0, 1.05))

    # ── ② Per-client Local CSE ───────────────────────────────────────────────
    display(Markdown("---\n#### ② Per-client Local CSE"))
    if local_cse:
        _plot(local_cse, "CSE (nats)",
              "Per-client Local Classification Semantic Entropy",
              "s", "local_cse",
              note="local model 的 Shannon 熵 H(p(y|x))。\n\n"
                   "- Benign：随训练收敛 → 熵**下降**\n"
                   "- Attacker (full flip)：可能同样低熵（confident but wrong）\n"
                   "- Partial / per-round random flip：attacker 熵通常**波动**且高于 benign")
    else:
        display(Markdown(
            "*`local_cse` 无数据。请确认已使用更新后的 `server.py` 运行实验。*"
        ))

    # ── ③ Trust Weights ──────────────────────────────────────────────────────
    display(Markdown("---\n#### ③ Per-client Trust Weights (HMP-GAE α)"))
    _plot(trust_w, "Aggregation Weight  α",
          "Per-client Trust Weights (HMP-GAE)",
          "^", "trust_weights",
          note="HMP-GAE 分配给每个 client 的聚合权重。"
               "**Attacker 应随轮次被压低；Benign 保持较高权重。**\n\n"
               "*FedAvg 模式下无此信号。*")

    # ── ④ Graph Structural Residual ──────────────────────────────────────────
    display(Markdown("---\n#### ④ Per-client Graph Structural Residual"))
    _plot(graph_res, "Graph Structural Residual",
          "Per-client HMP-GAE Graph Structural Residual",
          "D", "graph_residual",
          note="超图结构孤立信号（isolation score）：值越高 = 在超图中越孤立 = 越可疑。\n\n"
               "*FedAvg 模式下无此字段。*")

    # ── ⑤ Cosine Similarity ──────────────────────────────────────────────────
    display(Markdown("---\n#### ⑤ Per-client Cosine Similarity（update vs 加权平均）"))
    _plot(cosine_sim, "Cosine Similarity",
          "Per-client Cosine Similarity of Updates",
          "v", "cosine_similarity",
          note="每个 client 的 update 与加权平均 update 的 cosine similarity。"
               "**Attacker 的 update 方向与平均方向偏离较大（similarity 低或为负）。**")

    # ── ⑥ Euclidean Distance ─────────────────────────────────────────────────
    display(Markdown("---\n#### ⑥ Per-client Euclidean Distance（update vs 加权平均）"))
    _plot(euclid_dist, "Euclidean Distance",
          "Per-client Euclidean Distance of Updates",
          "p", "euclidean_distance",
          note="每个 client 的 update 与加权平均 update 的 L2 距离。"
               "**Attacker 的 update 通常与平均 update 距离较大。**")

    # ── ⑦ Acc vs CSE 散点图（最后一轮）─────────────────────────────────────
    display(Markdown("---\n#### ⑦ Accuracy vs CSE 散点图（最后一轮）"))
    if local_acc and local_cse:
        last_acc = {cid: v[-1] for cid, v in local_acc.items() if v}
        last_cse = {cid: v[-1] for cid, v in local_cse.items() if v}
        common   = sorted(set(last_acc) & set(last_cse))
        if common:
            fig, ax = plt.subplots(figsize=(6.5, 5))
            for cid in common:
                ax.scatter(last_acc[cid], last_cse[cid],
                           color=cmap.get(cid, "#666666"), s=110, zorder=3,
                           marker="X" if is_atk(cid) else "o",
                           edgecolors="white", linewidths=0.8)
                ax.annotate(c_label(cid), (last_acc[cid], last_cse[cid]),
                            textcoords="offset points", xytext=(6, 4), fontsize=8)
            ax.set_xlabel("Local Accuracy (last round)")
            ax.set_ylabel("Local CSE (nats, last round)")
            ax.set_title(f"Acc vs CSE — Last Round  [{exp}]")
            ax.grid(True, alpha=0.36, linestyle="--", linewidth=0.5)
            for s in ("top", "right", "bottom", "left"):
                ax.spines[s].set_visible(True)
            legend_h = [
                mlines.Line2D([], [], color=IEEE_BENIGN_COLORS[0], lw=0, marker="o",
                              markersize=9, markerfacecolor=IEEE_BENIGN_COLORS[0],
                              markeredgecolor="white", markeredgewidth=0.8, label="Benign"),
                mlines.Line2D([], [], color=IEEE_ATTACKER_COLORS[0], lw=0, marker="X",
                              markersize=10, markerfacecolor=IEEE_ATTACKER_COLORS[0],
                              markeredgecolor="white", markeredgewidth=0.8, label="Attacker"),
            ]
            ax.legend(handles=legend_h, loc="best",
                      framealpha=1.0, edgecolor="black")
            fig.tight_layout()
            display(Markdown(
                "Benign 应集中于**右下**（高 acc + 低熵 = 自信且正确）；\n\n"
                "Attacker (pairwise flip) 通常落在**左下**（低 acc + 低熵 = confident but wrong）。"
            ))
            out = FIG_DIR / f"{exp}__acc_vs_cse_lastround.png"
            fig.savefig(out)
            display(fig)
            plt.close(fig)
            display(Markdown(f"*Saved:* `{out}`"))
        else:
            display(Markdown("*acc 与 CSE 没有共同客户端，无法绘制散点图。*"))
    else:
        display(Markdown("*需同时有 `local_accuracies` 和 `local_cse` 数据。*"))


## Step 5: 数值明细（供复盘核查）

供精确复盘的结构化表格，与 Step 4a/4b 的图表一一对应：

- **① 完整 config**：可展开的 JSON widget，方便复制实验配置。
- **② 逐轮全局指标表**：Round × (clean_acc / global_loss / global_cse / agg_update_norm)。
- **③ 逐轮 per-client Trust Weights + Graph Residual 表**（HMP-GAE 模式）：行=Round，列=每个 client（标注 [ATK]/[BGN]）。
- **④ 逐轮 per-client Local Accuracy + CSE 表**：行=Round，列=client。
- **⑤ M7 多指标汇总**：final/best acc、mean CSE、PPL（若适用）。


In [ ]:
from __future__ import annotations
import json
from pathlib import Path
from IPython.display import display, Markdown, JSON

R = Path("results")
res_files = sorted(R.glob("*_results.json"), key=lambda p: p.stat().st_mtime, reverse=True)
if not res_files:
    print("未找到 results/*_results.json，请先跑 Step 3。")
else:
    rpath = res_files[0]
    with open(rpath, "r", encoding="utf-8") as f:
        data = json.load(f)
    cfg         = data.get("config") or {}
    exp         = cfg.get("experiment_name", "experiment")
    rounds_data = data.get("results") or []
    pm          = data.get("progressive_metrics") or {}
    atk_ids     = set(int(x) for x in (data.get("attacker_ids") or []))

    # fallback for old JSON without attacker_ids
    if not atk_ids:
        nc, na = int(cfg.get("num_clients", 0)), int(cfg.get("num_attackers", 0))
        if na > 0 and nc > 0:
            atk_ids = set(range(nc - na, nc))

    def _tag(cid):
        return "[ATK]" if int(cid) in atk_ids else "[BGN]"

    def _flt(v, d=6):
        if isinstance(v, float): return f"{v:.{d}f}"
        if isinstance(v, int):   return str(v)
        return str(v) if v is not None else "—"

    display(Markdown(f"### 结果文件: `{rpath.name}`"))

    # ── ① 完整 config ─────────────────────────────────────────────────────
    display(Markdown("### ① 完整 config（可展开）"))
    display(JSON(cfg))

    # ── ② 逐轮全局指标表 ──────────────────────────────────────────────────
    display(Markdown("### ② 逐轮全局指标"))
    hdr = "| Round | clean_acc | global_loss | global_cse | agg_update_norm |\n"
    hdr += "|---:|---:|---:|---:|---:|\n"
    rows = ""
    for r in rounds_data:
        agg  = r.get("aggregation") or {}
        rows += (f"| {r.get('round','')} "
                 f"| {_flt(r.get('clean_accuracy'))} "
                 f"| {_flt(r.get('global_loss'))} "
                 f"| {_flt(r.get('classification_semantic_entropy'))} "
                 f"| {_flt(agg.get('aggregated_update_norm'))} |\n")
    display(Markdown(hdr + rows))

    # -- (3) per-round per-client Trust-weight formula decomposition (NEW 2026-05-23, Issue 1) --
    # Every component of  s_i = -(w_g*z_g + alpha*z_r + w_s*z_s + beta*z_h)
    # plus combined-gate diagnostics (sus_z, gate) and the final aggregation alpha.
    # All read from r["aggregation"] (ordered by accepted_clients); re-ordered to a
    # stable client column layout. The 4 "Weighted ... Term" tables are reconstructed
    # as scalar_weight * z_signal so you can read directly which term dominates s.
    all_client_order = []  # from first round that has accepted_clients
    for r in rounds_data:
        co = (r.get("aggregation") or {}).get("accepted_clients") or []
        if co:
            all_client_order = [int(x) if str(x).isdigit() else x for x in co]
            break

    # (display title, aggregation key). Titles contain the extract_csvs
    # SUBHEADING_MAP key as a substring so each table becomes its own CSV.
    SIGNAL_TABLES = [
        ("Trust Weights alpha",        "trust_weights"),
        ("Graph Structural Residual",  "residual"),
        ("Graph Residual Zscore",      "graph_residual_z"),
        ("Recon Residual Raw",         "recon_residual"),
        ("Recon Residual Zscore",      "recon_residual_z"),
        ("Semantic Divergence Raw",    "sem_div"),
        ("Semantic Divergence Zscore", "sem_div_z"),
        ("Hist Deviation Raw",         "hist_dev"),
        ("Hist Deviation Zscore",      "hist_dev_z"),
        ("Trust Logit s",              "s"),
        ("Suspicion Zscore",           "sus_z"),
        ("Sigmoid Gate",               "gate"),
    ]
    # (display title, scalar-weight key, z-signal key) -> table of weight * z.
    WEIGHTED_TERMS = [
        ("Weighted Graph Term",     "graph_weight",               "graph_residual_z"),
        ("Weighted Recon Term",     "residual_weight_alpha",      "recon_residual_z"),
        ("Weighted Semantic Term",  "semantic_weight",            "sem_div_z"),
        ("Weighted Hist Term",      "hist_weight_beta_effective", "hist_dev_z"),
    ]

    def _collect(key):
        tbl = {}
        for r in rounds_data:
            agg = r.get("aggregation") or {}
            c_order = [int(x) if str(x).isdigit() else x for x in (agg.get("accepted_clients") or [])]
            vals = agg.get(key) or []
            if c_order and vals:
                idx = {cid: i for i, cid in enumerate(c_order)}
                tbl[r.get("round", "?")] = [
                    vals[idx[c]] if c in idx and idx[c] < len(vals) else None
                    for c in all_client_order
                ]
        return tbl

    def _collect_weighted(wkey, zkey):
        tbl = {}
        for r in rounds_data:
            agg = r.get("aggregation") or {}
            c_order = [int(x) if str(x).isdigit() else x for x in (agg.get("accepted_clients") or [])]
            zvals = agg.get(zkey) or []
            w = agg.get(wkey)
            if c_order and zvals and isinstance(w, (int, float)):
                idx = {cid: i for i, cid in enumerate(c_order)}
                tbl[r.get("round", "?")] = [
                    (w * zvals[idx[c]]) if c in idx and idx[c] < len(zvals) else None
                    for c in all_client_order
                ]
        return tbl

    def _render(title, tbl):
        if not tbl:
            return False
        col_tags = [f"C{c}{_tag(c)}" for c in all_client_order]
        hdr = "| Round | " + " | ".join(col_tags) + " |\n"
        hdr += "|---:|" + ":---:|" * len(col_tags) + "\n"
        body = ""
        for rnum, row in sorted(tbl.items()):
            body += f"| {rnum} | " + " | ".join(_flt(v, 4) for v in row) + " |\n"
        display(Markdown(f"**{title}**"))
        display(Markdown(hdr + body))
        return True

    is_hmp = cfg.get("defense_method", "").lower() == "hmp_gae"
    display(Markdown("### (3) per-round per-client Trust-weight decomposition"))
    if not is_hmp:
        display(Markdown("*defense_method is not `hmp_gae`; this section is N/A.*"))
    else:
        any_rendered = False
        for title, key in SIGNAL_TABLES:
            any_rendered = _render(title, _collect(key)) or any_rendered
        for title, wkey, zkey in WEIGHTED_TERMS:
            _render(title, _collect_weighted(wkey, zkey))
        if not any_rendered:
            display(Markdown("*No trust-signal data found (possibly FedAvg aggregation).*"))

    # ── ④ 逐轮 per-client Local Accuracy + CSE ───────────────────────────
    display(Markdown("### ④ 逐轮 per-client Local Accuracy + CSE"))

    # Use top-level fields (new format) or rebuild from per-round logs
    top_acc = data.get("local_accuracies") or {}
    top_cse = data.get("local_cse") or {}
    if not top_cse:
        for r in rounds_data:
            for raw, v in (r.get("local_cse") or {}).items():
                top_cse.setdefault(str(raw), []).append(v)

    all_cids_sorted = sorted(
        set(list(top_acc.keys()) + list(top_cse.keys())),
        key=lambda x: int(x) if str(x).isdigit() else 0
    )
    col_labels = [f"C{c}{_tag(c)}" for c in all_cids_sorted]

    def _pivot_table(series_dict, label):
        if not series_dict:
            display(Markdown(f"*{label} — 无数据*"))
            return
        # series_dict: {cid_str: [v_r0, v_r1, ...]}
        num_rounds = max(len(v) for v in series_dict.values()) if series_dict else 0
        hdr  = "| Round | " + " | ".join(col_labels) + " |\n"
        hdr += "|---:|" + ":---:|" * len(col_labels) + "\n"
        body = ""
        for ri in range(num_rounds):
            rnum = rounds_data[ri].get("round", ri + 1) if ri < len(rounds_data) else ri + 1
            cells = []
            for cid in all_cids_sorted:
                vals = series_dict.get(str(cid)) or series_dict.get(cid) or []
                cells.append(_flt(vals[ri], 4) if ri < len(vals) else "—")
            body += f"| {rnum} | " + " | ".join(cells) + " |\n"
        display(Markdown(f"**{label}**"))
        display(Markdown(hdr + body))

    _pivot_table(top_acc, "Local Accuracy")
    _pivot_table(top_cse, "Local CSE (nats)")

    # ── ⑤ M7 多指标汇总 ────────────────────────────────────────────────────
    display(Markdown("### ⑤ M7 多指标汇总"))
    ppl_path = R / f"{exp}_eval_ppl.json"

    # Raw PPL JSON (full eval_ppl content) — used to surface mean_nll and
    # per-class PPL spread directly in the M7 summary table.
    raw_ppl: dict = {}
    if ppl_path.is_file():
        try:
            with open(ppl_path, "r", encoding="utf-8") as f:
                raw_ppl = json.load(f) or {}
        except Exception:
            raw_ppl = {}

    try:
        from visualization import summarize_run_multi_metric
        s = summarize_run_multi_metric(
            rpath, ppl_json_path=ppl_path if ppl_path.is_file() else None
        )

        # Per-class spread metrics (computed from raw_ppl so we don't depend on
        # summarize_run_multi_metric exposing them).
        per_cls = raw_ppl.get("ppl_per_class", {}) or {}
        vals = [v for v in per_cls.values() if isinstance(v, (int, float))]
        if vals:
            pc_mean = sum(vals) / len(vals)
            pc_spread = max(vals) - min(vals)
            pc_std = (sum((v - pc_mean) ** 2 for v in vals) / len(vals)) ** 0.5
        else:
            pc_mean = pc_spread = pc_std = None

        rows = [
            ("final_clean_acc",      _flt(s.get("final_clean_acc"), 4)),
            ("best_clean_acc",       _flt(s.get("best_clean_acc"),  4)),
            ("final_cse",            _flt(s.get("final_cse"),       4)),
            ("mean_cse",             _flt(s.get("mean_cse"),        4)),
            ("**ppl_mean**",         f"**{_flt(s.get('ppl'), 2)}**" if s.get("ppl") else "—"),
            ("mean_nll",             _flt(raw_ppl.get("mean_nll"), 4) if raw_ppl else "—"),
            ("ppl_class_mean",       _flt(pc_mean,   2)),
            ("ppl_class_spread",     _flt(pc_spread, 2)),
            ("ppl_class_std",        _flt(pc_std,    2)),
            ("ppl_n_samples",        str(raw_ppl.get("n_samples", "—"))),
            ("ppl_dataset",          str(raw_ppl.get("dataset",   "—"))),
        ]
        md  = "| 指标 | 值 |\n|:---|:---|\n"
        md += "".join(f"| {k} | `{v}` |\n" for k, v in rows)
        display(Markdown(md))

        # Per-class PPL detail (when available)
        if per_cls:
            ag_names = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
            ds_name = (raw_ppl.get("dataset") or "").lower()
            md2  = "**Per-class PPL**\n\n| Class | Label | PPL | Δ vs class-mean |\n|---:|:---|---:|---:|\n"
            for k, v in sorted(per_cls.items(), key=lambda kv: int(kv[0])):
                label = ag_names.get(int(k), "—") if ds_name == "ag_news" else "—"
                if isinstance(v, (int, float)) and pc_mean is not None:
                    md2 += f"| {k} | {label} | {v:.2f} | {v - pc_mean:+.2f} |\n"
                else:
                    md2 += f"| {k} | {label} | — | — |\n"
            display(Markdown(md2))
    except Exception as e:
        display(Markdown(f"*M7 汇总异常: `{e}`*"))


## Step 6: 打包下载 results 产物

In [ ]:
import shutil
from pathlib import Path

out_dir = Path("results/hmp_gae_colab_bundle")
if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)
root = Path("results")
for p in list(root.rglob("*")):
    if p.is_file() and p.suffix in {".png", ".pdf", ".json", ".jsonl", ".csv"}:
        if "hmp_gae_colab_bundle" in p.parts:
            continue
        dest = out_dir / p.relative_to(root)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, dest)
zip_path = str(shutil.make_archive(str(out_dir), "zip", root_dir=out_dir))
print("Zip:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("非 Colab 或未自动下载, zip 路径:", zip_path, type(e).__name__)


## Step 7: 释放 GPU 并断开 Colab 运行时

**务必运行下一单元**，避免长时间占用 GPU。

In [ ]:
import gc, time
import torch

print("准备释放显存并断开本 Colab 运行时…")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    try:
        torch.cuda.synchronize()
    except Exception:
        pass
time.sleep(2)
try:
    from google.colab import runtime
    print("约 20 秒后 runtime.unassign() 释放实例。")
    time.sleep(20)
    runtime.unassign()
except Exception as e:
    print("非 Colab 或 unassign 不可用:", type(e).__name__, e)
    print("可手动: Runtime -> Disconnect and delete runtime")
